# Minecraft Server Control Panel - Colab Backend

Run your Minecraft server control panel backend on Google Colab with free GPU access!

## Features
- Free GPU/TPU access
- 12GB+ RAM
- Persistent storage (Google Drive)
- No cost limitations

## Setup Instructions
1. Mount Google Drive for persistent storage
2. Clone the repository
3. Install dependencies
4. Set up ngrok tunneling
5. Run the server

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

# Create directory for the project
!mkdir -p /content/drive/MyDrive/minecraft-control-panel
%cd /content/drive/MyDrive/minecraft-control-panel

In [ ]:
# Clone the repository
!git clone https://github.com/YehiaHS/minecraft-control-panel.git .

# Navigate to backend directory
%cd backend

In [ ]:
# Install Node.js and npm
!curl -fsSL https://deb.nodesource.com/setup_18.x | sudo -E bash -
!sudo apt-get install -y nodejs

# Verify installation
!node --version
!npm --version

In [ ]:
# Install backend dependencies (force reinstall for compatibility)
!rm -rf node_modules package-lock.json
!npm install

# Install ngrok for tunneling
!npm install -g ngrok

# Alternative: Install ngrok via apt
!sudo apt-get install -y ngrok

In [ ]:
# Set up ngrok authentication and clear problematic config
import getpass
import os

# Get ngrok token
ngrok_token = getpass.getpass('Enter your ngrok auth token: ')

# Clear any existing config that might cause reserved domain issues
os.system('rm -f ~/.ngrok2/ngrok.yml')

# Set authentication
!ngrok config add-authtoken {ngrok_token}

print("✅ ngrok authentication configured and config cleared")

In [ ]:
# Create environment file
env_content = """
NODE_ENV=production
PORT=3000
MINECRAFT_SERVER_PATH=./minecraft-server
ADMIN_USERNAME=admin
ADMIN_PASSWORD=colab_server_2025
"""

with open('.env', 'w') as f:
    f.write(env_content.strip())

print("✅ Environment file created!")
print("Default credentials:")
print("Username: admin")
print("Password: colab_server_2025")

In [ ]:
# Start ngrok tunnel with comprehensive error handling
import subprocess
import time
import signal
import os

def start_ngrok_tunnel():
    try:
        # Kill any existing ngrok processes
        os.system('pkill -f ngrok')
        time.sleep(2)
        
        print("🚀 Starting ngrok tunnel on port 3000...")
        
        # Start ngrok
        ngrok_process = subprocess.Popen(
            ['ngrok', 'http', '3000'], 
            stdout=subprocess.PIPE, 
            stderr=subprocess.PIPE,
            preexec_fn=os.setsid
        )
        
        # Wait for startup
        time.sleep(5)
        
        # Try to get URL with retries
        for attempt in range(5):
            try:
                result = subprocess.run(
                    ['curl', '-s', 'http://localhost:4040/api/tunnels'], 
                    capture_output=True, text=True, timeout=10
                )
                
                if result.returncode == 0 and result.stdout.strip():
                    import json
                    tunnels = json.loads(result.stdout)
                    if tunnels.get('tunnels') and len(tunnels['tunnels']) > 0:
                        public_url = tunnels['tunnels'][0]['public_url']
                        print(f"✅ SUCCESS: Backend accessible at: {public_url}")
                        print(f"📝 Copy this URL to your frontend configuration")
                        return public_url
                        
            except Exception as e:
                print(f"Attempt {attempt + 1} failed: {e}")
                time.sleep(3)
        
        print("❌ Could not get ngrok URL after 5 attempts")
        print("🔧 Troubleshooting:")
        print("1. Check your ngrok auth token is correct")
        print("2. Visit https://dashboard.ngrok.com for account status")
        print("3. Make sure you have available tunnel credits")
        return None
        
    except Exception as e:
        print(f"❌ Error starting ngrok: {e}")
        return None

# Start the tunnel
public_url = start_ngrok_tunnel()

In [ ]:
# Start the Minecraft control panel backend
if public_url:
    print("🎮 Starting Minecraft Server Control Panel Backend...")
    print(f"🌐 Public URL: {public_url}")
    print("⚠️  Keep this Colab tab open to keep the server running")
    print("💡 Frontend: https://yehiah.github.io/minecraft-control-panel")
    print()
    print("📋 Next steps:")
    print("1. Copy the public URL above")
    print("2. Go to the frontend and click 'Change Backend URL'")
    print("3. Paste the ngrok URL and connect")
    print()
    
    # Start the server
    !npm start
else:
    print("❌ Cannot start server - ngrok URL not available")
    print("🔧 Please fix ngrok issues above first")

In [ ]:
# ngrok Troubleshooting Cell
print("🔧 ngrok Troubleshooting:")

# Check if ngrok is running
print("\n1. Check ngrok process:")
!ps aux | grep ngrok

# Check ngrok status
print("\n2. Check ngrok web interface:")
!curl -s http://localhost:4040/api/tunnels || echo "ngrok not accessible"

# Check ngrok config
print("\n3. Check ngrok config:")
!ngrok config check 2>/dev/null || echo "No config found"

# Manual restart
print("\n4. Manual ngrok restart:")
!pkill -f ngrok
!ngrok http 3000 &
sleep 3
!curl -s http://localhost:4040/api/tunnels

## 🔧 Quick Fix for Node-Fetch Error

If you get an error about `node-fetch` being an ES Module, run this cell to fix it:

```python
# Fix node-fetch compatibility issue
%cd backend
!npm uninstall node-fetch
!npm install node-fetch@2.6.9
%cd ..
```

Then continue with the rest of the notebook.

## 🎯 How to Use

1. **Run all cells** in order from top to bottom
2. **Get ngrok URL** from the output
3. **Update frontend** to use your Colab backend URL
4. **Access control panel** at: https://yehiah.github.io/minecraft-control-panel

## 🔧 Troubleshooting

- **Session timeout**: Colab sessions timeout after ~12 hours of inactivity
- **Reconnect**: If disconnected, re-run the ngrok and server cells
- **GPU access**: Enable GPU in Runtime > Change runtime type
- **Storage**: Files are saved to Google Drive for persistence

## 💡 Pro Tips

- Keep Colab tab open to maintain server
- Use Google Drive for persistent Minecraft world storage
- Monitor ngrok URL for backend access
- Frontend demo works without backend for UI showcase